## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-22-11-50-20 +0000,bbc,Theresa May 'aghast' at Burnham's EU plans and...,https://www.bbc.co.uk/news/articles/cvj6l8y3x1...
1,2026-11-22-11-30-00 +0000,nypost,The Post’s 225 icons: George Washington’s surp...,https://nypost.com/2026/11/22/us-news/the-post...
2,2026-11-22-11-25-14 +0000,bbc,Russia-bound ship still on fire after attack i...,https://www.bbc.co.uk/news/articles/cj752ddve2...
3,2026-11-22-11-03-04 +0000,nyt,China’s Spies Set Sights on A.I. Chips Years B...,https://www.nytimes.com/2026/11/21/world/asia/...
4,2026-11-22-11-00-30 +0000,chitri,Loophole in Illinois law unleashes dog sales,https://www.chicagotribune.com/2026/11/22/illi...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2506/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
71,trump,21
104,new,8
83,california,8
49,he,8
253,dead,7


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
147,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...,57
152,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...,54
9,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...,48
49,2026-11-22-06-43-17 +0000,nypost,Mike Waltz warns Russia faces ‘unsustainable’ ...,https://nypost.com/2026/11/22/us-news/russia-f...,44
160,2026-11-21-15-10-50 +0000,chitri,Mexico detains top cartel boss ‘El Chapo Isidr...,https://www.chicagotribune.com/2026/11/21/el-c...,43


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
147,57,2026-11-21-16-32-51 +0000,nypost,Trump defends Stephen Miller after Hakeem Jeff...,https://nypost.com/2026/11/21/us-news/trump-de...
160,43,2026-11-21-15-10-50 +0000,chitri,Mexico detains top cartel boss ‘El Chapo Isidr...,https://www.chicagotribune.com/2026/11/21/el-c...
30,37,2026-11-22-10-00-00 +0000,wsj,Richer and more adventurous than their predece...,https://www.wsj.com/personal-finance/retiremen...
40,33,2026-11-22-08-43-29 +0000,nypost,Ex-White House press secretary Katie Zacharia ...,https://nypost.com/2026/11/22/us-news/ex-white...
139,33,2026-11-21-17-38-58 +0000,nypost,Russian cargo ship bursts into flames in Balti...,https://nypost.com/2026/11/21/world-news/russi...
59,31,2026-11-22-04-05-27 +0000,nypost,ICE officers arrest Alaska Airlines passengers...,https://nypost.com/2026/11/21/us-news/masked-i...
90,28,2026-11-21-23-50-12 +0000,chitri,Nations agree to extend talks on AI killer rob...,https://www.chicagotribune.com/2026/11/21/nati...
9,27,2026-11-22-11-00-00 +0000,wapo,Trump keeps losing in court. Here’s how he get...,https://www.washingtonpost.com/politics/2026/1...
35,27,2026-11-22-09-45-17 +0000,nypost,Fire at Indonesian candy factory leaves 3 dead...,https://nypost.com/2026/11/22/world-news/indon...
152,27,2026-11-21-16-11-17 +0000,nypost,Trump thinks there may be upside to reported c...,https://nypost.com/2026/11/21/us-news/trump-ta...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
